# Notebook 04: GPT Guided OWL ViT with Bounded Box Refinement

This notebook runs one final grounding method only:

```text
command and parsed target
    -> OWL ViT top candidate boxes
    -> GPT Vision selects the relationally correct candidate
    -> OWL ViT reruns inside an expanded local crop
    -> GPT Vision selects the best refined OWL ViT candidate
    -> GPT Vision reviews the selected boundary
    -> bounded edge adjustment, maximum 10 percent per edge
    -> final prediction and method-only metrics
```

Notebook 03 outputs are not loaded or compared. Notebook 04 writes to its own output folders.

## Model loading

This notebook uses the same simple OWL ViT and GPT Vision loading procedure as the previously working recursive fusion notebook. It does not use `snapshot_download`, a custom Hugging Face cache, `HF_TOKEN`, or direct model file downloads.


In [ ]:
# Colab dependency setup.
#
# This cell repairs the SymPy installation required by the preinstalled
# PyTorch build before Transformers imports `pipeline`.
#
# Run this notebook in a fresh Colab runtime from the top.

import os
import sys
import subprocess
import importlib
import importlib.metadata as metadata

# Disable the hf-xet transfer client before importing Transformers or
# huggingface_hub. This avoids the Xet path that previously stalled.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "900"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"


def torch_sympy_requirement():
    """Return the SymPy requirement declared by the installed PyTorch build."""
    try:
        requirements = metadata.requires("torch") or []
    except metadata.PackageNotFoundError:
        return "sympy==1.13.1"

    for requirement in requirements:
        if requirement.lower().startswith("sympy"):
            # Remove an optional environment marker such as:
            # sympy==1.13.1; python_version >= "3.9"
            return requirement.split(";", 1)[0].strip()

    return "sympy==1.13.1"


sympy_requirement = torch_sympy_requirement()
print("PyTorch SymPy requirement:", sympy_requirement)

# Repair SymPy first. Do not reinstall PyTorch because Colab supplies a
# CUDA-compatible build.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        sympy_requirement,
    ]
)

# Install the remaining notebook dependencies without replacing Colab's
# PyTorch installation.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
        "transformers",
        "accelerate",
        "huggingface_hub",
        "openai",
        "pandas",
        "numpy",
        "tqdm",
    ]
)

# Remove stale modules left by a previous failed import in the same runtime.
# A fresh runtime is still recommended.
for module_name in list(sys.modules):
    if (
        module_name == "sympy"
        or module_name.startswith("sympy.")
        or module_name == "torch"
        or module_name.startswith("torch.")
        or module_name == "transformers"
        or module_name.startswith("transformers.")
    ):
        sys.modules.pop(module_name, None)

importlib.invalidate_caches()

# Validate the repaired SymPy package before importing Transformers.
import sympy
from sympy import printing as sympy_printing

if not hasattr(sympy, "printing"):
    raise RuntimeError(
        "SymPy is still inconsistent. Restart the Colab runtime and run "
        "the notebook again from the first cell."
    )

print("SymPy version:", sympy.__version__)
print("SymPy location:", sympy.__file__)
print("SymPy printing module:", sympy_printing.__file__)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Drive is already mounted or this is not a Colab runtime.")

from pathlib import Path
import base64
import json
import shutil

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
from IPython.display import display

print("Pillow version:", Image.__version__)
print("HF Xet disabled:", os.environ.get("HF_HUB_DISABLE_XET"))
print("Dependency setup completed.")


In [ ]:
# Configuration. Edit PROJECT_ROOT manually if the project folder moves.

PROJECT_ROOT = Path('/content/drive/MyDrive/autonomous-delivery-robot/modules/perception/visual_grounding')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_ROOT}. Update PROJECT_ROOT above.')

OWL_MODEL_ID = 'google/owlvit-base-patch32'

# Persistent Hugging Face cache derived from the manually edited project root.
# Once downloaded successfully, later Colab runtimes can reuse the model.
HF_HOME_DIR = PROJECT_ROOT / 'models' / 'huggingface'
HF_HOME_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME_DIR)
GPT_MODEL = 'gpt-5'
GPT_DETAIL = 'high'

TOP_K_INITIAL = 5
TOP_K_REFINED = 5
OWL_RETRY_THRESHOLDS = [0.05, 0.02, 0.01, 0.005, 0.0]
MAX_ROWS = None
CLEAR_METHOD_OUTPUTS = True

# Local refinement around the candidate selected by GPT.
LOCAL_CROP_MARGIN = 0.25

# GPT may only make small final boundary changes.
MAX_GPT_EDGE_ADJUSTMENT = 0.10
MIN_GPT_ADJUSTMENT_CONFIDENCE = 0.70
MIN_ADJUSTED_BOX_OVERLAP = 0.60
MIN_ADJUSTED_AREA_RATIO = 0.60
MAX_ADJUSTED_AREA_RATIO = 1.50

SUCCESS_IOU_THRESHOLD = 0.25
STRICT_IOU_THRESHOLD = 0.50
WEAK_IOU_THRESHOLD = 0.10

BASE_BENCHMARK_PATH = PROJECT_ROOT / 'examples' / 'grounding_benchmark_from_ground_truth.csv'
PARSED_BENCHMARK_PATH = PROJECT_ROOT / 'examples' / 'grounding_benchmark_with_task_parser.csv'
GT_PATH = PROJECT_ROOT / 'examples' / 'ground_truth_boxes.csv'

METHOD_OUTPUT_DIR = PROJECT_ROOT / 'examples' / 'outputs' / 'gpt_guided_owlvit'
FINAL_RESULTS_PATH = METHOD_OUTPUT_DIR / 'gpt_guided_owlvit_results.csv'
METHOD_RESULTS_DIR = PROJECT_ROOT / 'results' / 'gpt_guided_owlvit'
INITIAL_CANDIDATE_DIR = METHOD_RESULTS_DIR / 'initial_candidates'
REFINEMENT_REGION_DIR = METHOD_RESULTS_DIR / 'refinement_regions'
REFINED_CANDIDATE_DIR = METHOD_RESULTS_DIR / 'refined_candidates'
PRE_ADJUSTMENT_DIR = METHOD_RESULTS_DIR / 'pre_adjustment'
EDGE_REVIEW_DIR = METHOD_RESULTS_DIR / 'edge_review'
FINAL_PREDICTION_DIR = METHOD_RESULTS_DIR / 'final_predictions'
METRICS_DIR = METHOD_RESULTS_DIR / 'metrics'

METHOD_DIRS = [
    METHOD_OUTPUT_DIR,
    INITIAL_CANDIDATE_DIR,
    REFINEMENT_REGION_DIR,
    REFINED_CANDIDATE_DIR,
    PRE_ADJUSTMENT_DIR,
    EDGE_REVIEW_DIR,
    FINAL_PREDICTION_DIR,
    METRICS_DIR,
]

for folder in METHOD_DIRS:
    folder.mkdir(parents=True, exist_ok=True)

if CLEAR_METHOD_OUTPUTS:
    for folder in METHOD_DIRS:
        for item in folder.iterdir():
            if item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)

print('Project root:', PROJECT_ROOT)
print('Method output CSV folder:', METHOD_OUTPUT_DIR)
print('Method result folder:', METHOD_RESULTS_DIR)

In [ ]:
# Load the current benchmark and corrected ground truth.

benchmark_path = PARSED_BENCHMARK_PATH

if not benchmark_path.exists():
    raise FileNotFoundError(
        'Parsed grounding benchmark not found. Run Notebook 02, then Notebook 03 before Notebook 04: '
        + str(PARSED_BENCHMARK_PATH)
    )
if not GT_PATH.exists():
    raise FileNotFoundError('ground_truth_boxes.csv was not found. Run Notebook 02 first.')

benchmark = pd.read_csv(benchmark_path)
ground_truth = pd.read_csv(GT_PATH)

required_benchmark_columns = ['image_id', 'image_path', 'grounding_prompt', 'target_object']
missing_benchmark_columns = [c for c in required_benchmark_columns if c not in benchmark.columns]
if missing_benchmark_columns:
    raise ValueError(f'Benchmark is missing columns: {missing_benchmark_columns}')

required_gt_columns = ['image_id', 'gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max']
missing_gt_columns = [c for c in required_gt_columns if c not in ground_truth.columns]
if missing_gt_columns:
    raise ValueError(f'Ground truth CSV is missing columns: {missing_gt_columns}')

benchmark['image_id'] = benchmark['image_id'].astype(str)
ground_truth['image_id'] = ground_truth['image_id'].astype(str)

if benchmark['image_id'].duplicated().any():
    raise ValueError('Parsed benchmark contains duplicate image IDs.')
if ground_truth['image_id'].duplicated().any():
    raise ValueError('Ground truth CSV contains duplicate image IDs.')

benchmark_ids = set(benchmark['image_id'])
ground_truth_ids = set(ground_truth['image_id'])
if benchmark_ids != ground_truth_ids:
    raise ValueError(
        'Notebook 04 input ID mismatch. '
        f'benchmark-only={sorted(benchmark_ids - ground_truth_ids)[:10]}, '
        f'ground-truth-only={sorted(ground_truth_ids - benchmark_ids)[:10]}'
    )

coordinate_columns = ['gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max']
benchmark = benchmark.drop(columns=[c for c in coordinate_columns if c in benchmark.columns], errors='ignore')
benchmark = benchmark.merge(
    ground_truth[['image_id'] + coordinate_columns],
    on='image_id',
    how='inner',
)

# Notebook 04 can run after Notebook 02 even when Notebook 03 has not created parsed fields.
if 'source_command' not in benchmark.columns:
    if 'command' in benchmark.columns:
        benchmark['source_command'] = benchmark['command']
    else:
        benchmark['source_command'] = benchmark['grounding_prompt']
if 'parsed_target_object' not in benchmark.columns:
    benchmark['parsed_target_object'] = benchmark['target_object']
if 'parsed_location_hint' not in benchmark.columns:
    benchmark['parsed_location_hint'] = ''
if 'model_grounding_prompt' not in benchmark.columns:
    benchmark['model_grounding_prompt'] = benchmark['grounding_prompt']
if 'parser_candidate_prompts' not in benchmark.columns:
    benchmark['parser_candidate_prompts'] = ''

valid_rows = []
skipped_rows = []

for row in benchmark.to_dict('records'):
    image_path = PROJECT_ROOT / str(row['image_path'])
    if not image_path.exists():
        skipped_rows.append({'image_id': row['image_id'], 'reason': 'raw image missing', 'image_path': row['image_path']})
        continue

    try:
        for column in coordinate_columns:
            row[column] = float(row[column])
    except Exception:
        skipped_rows.append({'image_id': row['image_id'], 'reason': 'invalid ground truth coordinates', 'image_path': row['image_path']})
        continue

    valid_rows.append(row)

benchmark = pd.DataFrame(valid_rows)
if MAX_ROWS is not None:
    benchmark = benchmark.head(int(MAX_ROWS)).copy()

print('Benchmark source:', benchmark_path)
print('Rows used:', len(benchmark))
print('Rows skipped:', len(skipped_rows))

if skipped_rows:
    skipped_path = METHOD_RESULTS_DIR / 'skipped_rows.csv'
    pd.DataFrame(skipped_rows).to_csv(skipped_path, index=False)
    display(pd.DataFrame(skipped_rows).head(20))

if benchmark.empty:
    raise RuntimeError('No valid rows remain after benchmark and image validation.')

display(benchmark.head())

In [ ]:
# Geometry, prompt, drawing, and API helpers.

RELATION_TERMS = [
    'beside', 'next to', 'near', 'between', 'behind', 'in front of',
    'closest', 'left of', 'right of', 'under', 'above', 'below',
    'blocked by', 'partly blocked', 'by the', 'on the',
]


def safe_text(value):
    if value is None:
        return ''
    text = str(value).strip()
    return '' if text.lower() in {'nan', 'none', 'null'} else text


def unique_keep_order(values):
    output = []
    seen = set()
    for value in values:
        text = safe_text(value)
        if not text:
            continue
        key = text.lower()
        if key not in seen:
            output.append(text)
            seen.add(key)
    return output


def is_relational_prompt(row):
    text = ' '.join(
        safe_text(row.get(column, ''))
        for column in ['source_command', 'parsed_location_hint', 'grounding_prompt', 'filename_description']
    ).lower()
    return any(term in text for term in RELATION_TERMS)


def normalize_box_xyxy(box, image_size):
    if box is None:
        return None
    try:
        x1, y1, x2, y2 = [float(value) for value in box]
    except Exception:
        return None
    if not all(np.isfinite([x1, y1, x2, y2])):
        return None

    width, height = image_size
    if max(abs(x1), abs(y1), abs(x2), abs(y2)) <= 1.5:
        x1 *= width
        x2 *= width
        y1 *= height
        y2 *= height

    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1

    x1 = max(0.0, min(float(width - 1), x1))
    x2 = max(0.0, min(float(width - 1), x2))
    y1 = max(0.0, min(float(height - 1), y1))
    y2 = max(0.0, min(float(height - 1), y2))

    if x2 - x1 <= 1 or y2 - y1 <= 1:
        return None
    return [x1, y1, x2, y2]


def box_area(box):
    if box is None:
        return 0.0
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])


def box_iou_xyxy(box_a, box_b):
    if box_a is None or box_b is None:
        return 0.0
    ax1, ay1, ax2, ay2 = [float(v) for v in box_a]
    bx1, by1, bx2, by2 = [float(v) for v in box_b]

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    denominator = box_area(box_a) + box_area(box_b) - intersection
    return float(intersection / denominator) if denominator > 0 else 0.0


def grade_iou(iou, status):
    if status != 'success':
        return 'failed'
    if iou >= STRICT_IOU_THRESHOLD:
        return 'strict_success'
    if iou >= SUCCESS_IOU_THRESHOLD:
        return 'success'
    if iou >= WEAK_IOU_THRESHOLD:
        return 'weak_overlap'
    return 'poor'


def expand_box(box, margin_fraction, image_size):
    if box is None:
        return None
    x1, y1, x2, y2 = box
    width = x2 - x1
    height = y2 - y1
    expanded = [
        x1 - width * margin_fraction,
        y1 - height * margin_fraction,
        x2 + width * margin_fraction,
        y2 + height * margin_fraction,
    ]
    return normalize_box_xyxy(expanded, image_size)


def crop_box_to_full_image(local_box, crop_box):
    return [
        crop_box[0] + local_box[0],
        crop_box[1] + local_box[1],
        crop_box[0] + local_box[2],
        crop_box[1] + local_box[3],
    ]


def safe_candidate_index(value, candidate_count):
    try:
        index = int(value)
    except Exception:
        return None
    return index if 1 <= index <= candidate_count else None


def image_to_data_url(image_path):
    image_path = Path(image_path)
    mime = 'image/png' if image_path.suffix.lower() == '.png' else 'image/jpeg'
    encoded = base64.b64encode(image_path.read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{encoded}'


def extract_json_object(text):
    text = safe_text(text)
    start = text.find('{')
    end = text.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('No JSON object found in GPT response.')
    return json.loads(text[start:end + 1])


def candidate_labels_for_row(row):
    values = [
        row.get('parsed_target_object', ''),
        row.get('target_object', ''),
        row.get('model_grounding_prompt', ''),
        row.get('grounding_prompt', ''),
    ]
    parser_candidates = safe_text(row.get('parser_candidate_prompts', ''))
    if parser_candidates:
        values.extend(part.strip() for part in parser_candidates.split('|'))

    target = safe_text(row.get('parsed_target_object', row.get('target_object', ''))).lower()
    if target == 'package':
        values.extend(['package', 'box', 'cardboard box', 'delivery package', 'parcel'])
    elif target == 'person':
        values.extend(['person', 'human'])
    elif target == 'door':
        values.extend(['door', 'apartment door'])
    elif target == 'elevator button panel':
        values.extend(['elevator button panel', 'button panel', 'elevator button'])

    return unique_keep_order(values)[:12]


def draw_numbered_candidates(image, candidates, output_path, title):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    draw.rectangle([0, 0, canvas.width, 44], fill='black')
    draw.text((10, 13), title, fill='white')

    for index, candidate in enumerate(candidates, start=1):
        box = [int(round(value)) for value in candidate['box']]
        draw.rectangle(box, outline='blue', width=7)
        label_x = box[0]
        label_y = max(45, box[1] - 28)
        draw.rectangle([label_x, label_y, label_x + 68, label_y + 25], fill='blue')
        draw.text((label_x + 5, label_y + 5), f'C{index}', fill='white')

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(output_path)
    return output_path


def draw_region(image, region, output_path, title):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    draw.rectangle([0, 0, canvas.width, 44], fill='black')
    draw.text((10, 13), title, fill='white')
    draw.rectangle([int(round(v)) for v in region], outline='yellow', width=8)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(output_path)
    return output_path


def draw_selected_box(image, box, output_path, title, color='cyan'):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    draw.rectangle([0, 0, canvas.width, 44], fill='black')
    draw.text((10, 13), title, fill='white')
    draw.rectangle([int(round(v)) for v in box], outline=color, width=8)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(output_path)
    return output_path


def create_edge_review_crop(image, box, output_path):
    review_region = expand_box(box, 0.35, image.size)
    integer_region = [int(round(v)) for v in review_region]
    crop = image.crop(tuple(integer_region))

    local_box = [
        box[0] - review_region[0],
        box[1] - review_region[1],
        box[2] - review_region[0],
        box[3] - review_region[1],
    ]
    draw = ImageDraw.Draw(crop)
    draw.rectangle([int(round(v)) for v in local_box], outline='cyan', width=8)
    draw.rectangle([0, 0, crop.width, 38], fill='black')
    draw.text((8, 10), 'Selected OWL ViT box for bounded edge review', fill='white')

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    crop.save(output_path)
    return output_path


def apply_bounded_adjustment(box, decision, image_size):
    decision_type = safe_text(decision.get('decision', '')).lower()
    confidence = float(decision.get('confidence', 0.0) or 0.0)

    if decision_type != 'adjust' or confidence < MIN_GPT_ADJUSTMENT_CONFIDENCE:
        return box, False, 'not_applied'

    try:
        left_shift = float(decision.get('left_shift', 0.0) or 0.0)
        top_shift = float(decision.get('top_shift', 0.0) or 0.0)
        right_shift = float(decision.get('right_shift', 0.0) or 0.0)
        bottom_shift = float(decision.get('bottom_shift', 0.0) or 0.0)
    except Exception:
        return box, False, 'invalid_shift_values'

    shifts = [left_shift, top_shift, right_shift, bottom_shift]
    shifts = [max(-MAX_GPT_EDGE_ADJUSTMENT, min(MAX_GPT_EDGE_ADJUSTMENT, value)) for value in shifts]
    left_shift, top_shift, right_shift, bottom_shift = shifts

    x1, y1, x2, y2 = box
    width = x2 - x1
    height = y2 - y1

    proposed = normalize_box_xyxy(
        [
            x1 + left_shift * width,
            y1 + top_shift * height,
            x2 + right_shift * width,
            y2 + bottom_shift * height,
        ],
        image_size,
    )
    if proposed is None:
        return box, False, 'invalid_adjusted_box'

    overlap = box_iou_xyxy(box, proposed)
    original_area = box_area(box)
    adjusted_area_ratio = box_area(proposed) / original_area if original_area > 0 else 0.0

    if overlap < MIN_ADJUSTED_BOX_OVERLAP:
        return box, False, 'adjustment_overlap_gate_failed'
    if not (MIN_ADJUSTED_AREA_RATIO <= adjusted_area_ratio <= MAX_ADJUSTED_AREA_RATIO):
        return box, False, 'adjustment_area_gate_failed'

    return proposed, True, 'applied'


In [ ]:
# Load OWL ViT and the GPT Vision client.
#
# The bounded-refinement method is unchanged. This cell validates SymPy before
# importing Transformers so a broken PyTorch dependency is detected clearly.

import os
import sympy

if not hasattr(sympy, "printing"):
    raise RuntimeError(
        "SymPy is not loaded correctly. Restart the Colab runtime and run "
        "the notebook from the first cell."
    )

import torch
import transformers
import huggingface_hub
from transformers import pipeline
from huggingface_hub import login
from openai import OpenAI

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(names, required=True):
    """Read the first available environment variable or Colab secret."""
    if isinstance(names, str):
        names = [names]

    for name in names:
        value = os.environ.get(name)
        if value:
            return value

        if userdata is not None:
            try:
                value = userdata.get(name)
            except Exception:
                value = None

            if value:
                os.environ[name] = value
                return value

    if required:
        expected = " or ".join(names)
        raise EnvironmentError(
            f"{expected} is missing. Add it to Colab Secrets, enable notebook "
            "access, then rerun this cell."
        )

    return None


openai_key = read_secret("OPENAI_API_KEY")
hf_token = read_secret(["HF_TOKEN", "HF_TOKEN_COLAB"])

login(
    token=hf_token,
    add_to_git_credential=False,
    skip_if_logged_in=False,
)

print("torch version:", torch.__version__)
print("sympy version:", sympy.__version__)
print("transformers version:", transformers.__version__)
print("huggingface_hub version:", huggingface_hub.__version__)
print("Hugging Face cache:", os.environ.get("HF_HOME"))
print("Hugging Face authentication: active")

client = OpenAI(api_key=openai_key)
device_id = 0 if torch.cuda.is_available() else -1

print("Loading OWL ViT...")

owl_detector = pipeline(
    task="zero-shot-object-detection",
    model=OWL_MODEL_ID,
    device=device_id,
    token=hf_token,
    model_kwargs={
        "use_safetensors": False,
    },
)

print("OWL ViT and GPT Vision client are ready.")


In [ ]:
# OWL ViT candidate generation and GPT Vision decisions.


def owlvit_top_candidates(image, labels, top_k):
    labels = unique_keep_order(labels) or ['object']
    all_predictions = []

    for threshold in OWL_RETRY_THRESHOLDS:
        try:
            predictions = owl_detector(
                image,
                candidate_labels=labels,
                threshold=float(threshold),
            )
        except TypeError:
            predictions = owl_detector(image, labels, threshold=float(threshold))

        for prediction in predictions or []:
            raw_box = prediction.get('box', {})
            box = normalize_box_xyxy(
                [
                    raw_box.get('xmin'),
                    raw_box.get('ymin'),
                    raw_box.get('xmax'),
                    raw_box.get('ymax'),
                ],
                image.size,
            )
            if box is None:
                continue
            all_predictions.append({
                'box': box,
                'score': float(prediction.get('score', 0.0)),
                'label': safe_text(prediction.get('label', '')),
                'threshold': float(threshold),
            })

        if all_predictions:
            break

    all_predictions.sort(key=lambda item: item['score'], reverse=True)

    kept = []
    for candidate in all_predictions:
        if any(box_iou_xyxy(candidate['box'], existing['box']) > 0.85 for existing in kept):
            continue
        kept.append(candidate)
        if len(kept) >= top_k:
            break
    return kept


def gpt_choose_candidate_or_region(visual_image_path, row, candidates, image_size, allow_region):
    candidate_lines = []
    for index, candidate in enumerate(candidates, start=1):
        candidate_lines.append(
            f"C{index}: label={candidate['label']}, score={candidate['score']:.4f}, "
            f"box={[round(v, 1) for v in candidate['box']]}"
        )

    candidate_description = '\n'.join(candidate_lines) if candidate_lines else 'No OWL ViT candidates were found.'

    instruction = f'''Return one JSON object only.

Image size: width={image_size[0]}, height={image_size[1]}
Original command: {safe_text(row.get('source_command', row.get('grounding_prompt', '')))}
Target object: {safe_text(row.get('parsed_target_object', row.get('target_object', 'object')))}
Location hint: {safe_text(row.get('parsed_location_hint', ''))}
Relational prompt: {is_relational_prompt(row)}

The supplied image shows numbered blue OWL ViT candidates when candidates exist.
Candidates:
{candidate_description}

Rules:
1. Select the candidate that best satisfies the full command, especially its spatial relation.
2. Do not choose only by the highest detector score.
3. If one candidate is correct, return status="selected" and its 1-based candidate number.
4. If no candidate is suitable and allow_region={allow_region}, return status="refine" and a coarse pixel region [x_min, y_min, x_max, y_max].
5. Otherwise return status="failed".

Schema:
{{
  "status": "selected" or "refine" or "failed",
  "selected_candidate": integer or null,
  "confidence": number from 0 to 1,
  "relation_match": true or false,
  "region": [x_min, y_min, x_max, y_max] or null,
  "reason": "short explanation"
}}'''

    response = client.responses.create(
        model=GPT_MODEL,
        input=[{
            'role': 'user',
            'content': [
                {'type': 'input_text', 'text': instruction},
                {
                    'type': 'input_image',
                    'image_url': image_to_data_url(visual_image_path),
                    'detail': GPT_DETAIL,
                },
            ],
        }],
    )
    return extract_json_object(response.output_text)


def gpt_review_selected_box(edge_review_path, row):
    instruction = f'''Return one JSON object only.

The cyan box in the supplied crop is the candidate already selected as the correct target.
Do not select another object. Review only whether the four box edges tightly contain the target.

Original command: {safe_text(row.get('source_command', row.get('grounding_prompt', '')))}
Target object: {safe_text(row.get('parsed_target_object', row.get('target_object', 'object')))}
Location hint: {safe_text(row.get('parsed_location_hint', ''))}

Each shift is a fraction of the current box width or height and must be between -{MAX_GPT_EDGE_ADJUSTMENT:.2f} and {MAX_GPT_EDGE_ADJUSTMENT:.2f}.
Shift meanings:
- left_shift: positive moves the left edge right, negative moves it left.
- top_shift: positive moves the top edge down, negative moves it up.
- right_shift: positive moves the right edge right, negative moves it left.
- bottom_shift: positive moves the bottom edge down, negative moves it up.

Use decision="accept" when the box is already suitable.
Use decision="adjust" only for a small boundary correction.
Use decision="uncertain" when the boundary cannot be judged reliably.

Schema:
{{
  "decision": "accept" or "adjust" or "uncertain",
  "left_shift": number,
  "top_shift": number,
  "right_shift": number,
  "bottom_shift": number,
  "confidence": number from 0 to 1,
  "reason": "short explanation"
}}'''

    response = client.responses.create(
        model=GPT_MODEL,
        input=[{
            'role': 'user',
            'content': [
                {'type': 'input_text', 'text': instruction},
                {
                    'type': 'input_image',
                    'image_url': image_to_data_url(edge_review_path),
                    'detail': GPT_DETAIL,
                },
            ],
        }],
    )
    return extract_json_object(response.output_text)


In [ ]:
# Run the single GPT-guided OWL ViT method.

final_rows = []
initial_candidate_rows = []
refined_candidate_rows = []
selection_decision_rows = []
edge_adjustment_rows = []

for row in tqdm(benchmark.to_dict('records'), desc='GPT guided OWL ViT bounded refinement'):
    image_id = str(row['image_id'])
    image_path = PROJECT_ROOT / str(row['image_path'])
    image = Image.open(image_path).convert('RGB')
    image_size = image.size
    labels = candidate_labels_for_row(row)

    ground_truth_box = normalize_box_xyxy(
        [row['gt_x_min'], row['gt_y_min'], row['gt_x_max'], row['gt_y_max']],
        image_size,
    )

    # Stage 1: full-image OWL ViT candidates and GPT relational selection.
    initial_candidates = owlvit_top_candidates(image, labels, TOP_K_INITIAL)
    initial_visual_path = INITIAL_CANDIDATE_DIR / f'{image_id}.png'
    draw_numbered_candidates(image, initial_candidates, initial_visual_path, 'Initial OWL ViT candidates')

    for candidate_index, candidate in enumerate(initial_candidates, start=1):
        initial_candidate_rows.append({
            'image_id': image_id,
            'candidate_id': candidate_index,
            'label': candidate['label'],
            'score': candidate['score'],
            'threshold': candidate['threshold'],
            'x_min': candidate['box'][0],
            'y_min': candidate['box'][1],
            'x_max': candidate['box'][2],
            'y_max': candidate['box'][3],
        })

    try:
        initial_decision = gpt_choose_candidate_or_region(
            initial_visual_path if initial_candidates else image_path,
            row,
            initial_candidates,
            image_size,
            allow_region=True,
        )
    except Exception as exc:
        initial_decision = {
            'status': 'failed',
            'selected_candidate': None,
            'confidence': 0.0,
            'relation_match': False,
            'region': None,
            'reason': str(exc)[:500],
        }

    selection_decision_rows.append({'image_id': image_id, 'stage': 'initial', **initial_decision})

    initial_index = safe_candidate_index(initial_decision.get('selected_candidate'), len(initial_candidates))
    initial_selected = None
    refinement_seed = None
    refinement_source = ''

    if initial_decision.get('status') == 'selected' and initial_index is not None:
        initial_selected = initial_candidates[initial_index - 1]
        refinement_seed = initial_selected['box']
        refinement_source = 'gpt_selected_initial_candidate'
    elif initial_decision.get('status') == 'refine':
        refinement_seed = normalize_box_xyxy(initial_decision.get('region'), image_size)
        refinement_source = 'gpt_coarse_region'

    if refinement_seed is None:
        final_rows.append({
            'image_id': image_id,
            'image_path': row['image_path'],
            'status': 'failed',
            'failure_stage': 'initial_selection',
            'relational_prompt': is_relational_prompt(row),
            'initial_candidate_count': len(initial_candidates),
            'refined_candidate_count': 0,
            'gpt_selection_confidence': initial_decision.get('confidence', np.nan),
            'relation_match': initial_decision.get('relation_match', False),
            'adjustment_applied': False,
            'pred_x_min': np.nan,
            'pred_y_min': np.nan,
            'pred_x_max': np.nan,
            'pred_y_max': np.nan,
            'gt_x_min': ground_truth_box[0] if ground_truth_box else np.nan,
            'gt_y_min': ground_truth_box[1] if ground_truth_box else np.nan,
            'gt_x_max': ground_truth_box[2] if ground_truth_box else np.nan,
            'gt_y_max': ground_truth_box[3] if ground_truth_box else np.nan,
            'iou': 0.0,
            'bbox_quality': 'failed',
            'notes': safe_text(initial_decision.get('reason', '')),
        })
        continue

    # Stage 2: rerun OWL ViT in an expanded crop around the selected candidate or GPT region.
    refinement_region = expand_box(refinement_seed, LOCAL_CROP_MARGIN, image_size)
    region_visual_path = REFINEMENT_REGION_DIR / f'{image_id}.png'
    draw_region(image, refinement_region, region_visual_path, f'Refinement region: {refinement_source}')

    integer_region = [int(round(v)) for v in refinement_region]
    crop = image.crop(tuple(integer_region))
    local_candidates = owlvit_top_candidates(crop, labels, TOP_K_REFINED)

    refined_candidates = []
    for candidate_index, local_candidate in enumerate(local_candidates, start=1):
        full_box = normalize_box_xyxy(
            crop_box_to_full_image(local_candidate['box'], refinement_region),
            image_size,
        )
        if full_box is None:
            continue
        refined_candidate = {**local_candidate, 'box': full_box}
        refined_candidates.append(refined_candidate)
        refined_candidate_rows.append({
            'image_id': image_id,
            'candidate_id': candidate_index,
            'label': refined_candidate['label'],
            'score': refined_candidate['score'],
            'threshold': refined_candidate['threshold'],
            'region_x_min': refinement_region[0],
            'region_y_min': refinement_region[1],
            'region_x_max': refinement_region[2],
            'region_y_max': refinement_region[3],
            'x_min': full_box[0],
            'y_min': full_box[1],
            'x_max': full_box[2],
            'y_max': full_box[3],
        })

    refined_visual_path = REFINED_CANDIDATE_DIR / f'{image_id}.png'
    draw_numbered_candidates(image, refined_candidates, refined_visual_path, 'Locally refined OWL ViT candidates')

    selected_candidate = None
    final_selection_decision = initial_decision
    selection_stage = ''

    if refined_candidates:
        try:
            refined_decision = gpt_choose_candidate_or_region(
                refined_visual_path,
                row,
                refined_candidates,
                image_size,
                allow_region=False,
            )
        except Exception as exc:
            refined_decision = {
                'status': 'failed',
                'selected_candidate': None,
                'confidence': 0.0,
                'relation_match': False,
                'region': None,
                'reason': str(exc)[:500],
            }

        selection_decision_rows.append({'image_id': image_id, 'stage': 'refined', **refined_decision})
        refined_index = safe_candidate_index(refined_decision.get('selected_candidate'), len(refined_candidates))
        if refined_decision.get('status') == 'selected' and refined_index is not None:
            selected_candidate = refined_candidates[refined_index - 1]
            final_selection_decision = refined_decision
            selection_stage = 'gpt_selected_refined_candidate'

    # If local OWL ViT finds no better candidate, retain the GPT-selected initial candidate.
    if selected_candidate is None and initial_selected is not None:
        selected_candidate = initial_selected
        selection_stage = 'initial_candidate_retained_after_local_refinement'

    if selected_candidate is None:
        final_rows.append({
            'image_id': image_id,
            'image_path': row['image_path'],
            'status': 'failed',
            'failure_stage': 'refined_selection',
            'relational_prompt': is_relational_prompt(row),
            'initial_candidate_count': len(initial_candidates),
            'refined_candidate_count': len(refined_candidates),
            'gpt_selection_confidence': final_selection_decision.get('confidence', np.nan),
            'relation_match': final_selection_decision.get('relation_match', False),
            'adjustment_applied': False,
            'pred_x_min': np.nan,
            'pred_y_min': np.nan,
            'pred_x_max': np.nan,
            'pred_y_max': np.nan,
            'gt_x_min': ground_truth_box[0] if ground_truth_box else np.nan,
            'gt_y_min': ground_truth_box[1] if ground_truth_box else np.nan,
            'gt_x_max': ground_truth_box[2] if ground_truth_box else np.nan,
            'gt_y_max': ground_truth_box[3] if ground_truth_box else np.nan,
            'iou': 0.0,
            'bbox_quality': 'failed',
            'notes': safe_text(final_selection_decision.get('reason', '')),
        })
        continue

    pre_adjustment_box = selected_candidate['box']
    draw_selected_box(
        image,
        pre_adjustment_box,
        PRE_ADJUSTMENT_DIR / f'{image_id}.png',
        f'Pre-adjustment selected box: {selection_stage}',
        color='orange',
    )

    # Stage 3: GPT reviews only the selected box edges and may make a bounded correction.
    edge_review_path = create_edge_review_crop(
        image,
        pre_adjustment_box,
        EDGE_REVIEW_DIR / f'{image_id}.png',
    )

    try:
        edge_decision = gpt_review_selected_box(edge_review_path, row)
    except Exception as exc:
        edge_decision = {
            'decision': 'uncertain',
            'left_shift': 0.0,
            'top_shift': 0.0,
            'right_shift': 0.0,
            'bottom_shift': 0.0,
            'confidence': 0.0,
            'reason': str(exc)[:500],
        }

    final_box, adjustment_applied, adjustment_gate = apply_bounded_adjustment(
        pre_adjustment_box,
        edge_decision,
        image_size,
    )

    edge_adjustment_rows.append({
        'image_id': image_id,
        **edge_decision,
        'adjustment_applied': adjustment_applied,
        'adjustment_gate': adjustment_gate,
        'pre_x_min': pre_adjustment_box[0],
        'pre_y_min': pre_adjustment_box[1],
        'pre_x_max': pre_adjustment_box[2],
        'pre_y_max': pre_adjustment_box[3],
        'final_x_min': final_box[0],
        'final_y_min': final_box[1],
        'final_x_max': final_box[2],
        'final_y_max': final_box[3],
    })

    iou = box_iou_xyxy(final_box, ground_truth_box)
    status = 'success'

    final_rows.append({
        'image_id': image_id,
        'image_path': row['image_path'],
        'status': status,
        'failure_stage': '',
        'selection_stage': selection_stage,
        'refinement_source': refinement_source,
        'relational_prompt': is_relational_prompt(row),
        'initial_candidate_count': len(initial_candidates),
        'refined_candidate_count': len(refined_candidates),
        'selected_label': selected_candidate['label'],
        'owl_score': selected_candidate['score'],
        'gpt_selection_confidence': final_selection_decision.get('confidence', np.nan),
        'relation_match': final_selection_decision.get('relation_match', False),
        'gpt_edge_decision': edge_decision.get('decision', ''),
        'gpt_edge_confidence': edge_decision.get('confidence', np.nan),
        'adjustment_applied': adjustment_applied,
        'adjustment_gate': adjustment_gate,
        'pre_adjustment_x_min': pre_adjustment_box[0],
        'pre_adjustment_y_min': pre_adjustment_box[1],
        'pre_adjustment_x_max': pre_adjustment_box[2],
        'pre_adjustment_y_max': pre_adjustment_box[3],
        'pred_x_min': final_box[0],
        'pred_y_min': final_box[1],
        'pred_x_max': final_box[2],
        'pred_y_max': final_box[3],
        'gt_x_min': ground_truth_box[0] if ground_truth_box else np.nan,
        'gt_y_min': ground_truth_box[1] if ground_truth_box else np.nan,
        'gt_x_max': ground_truth_box[2] if ground_truth_box else np.nan,
        'gt_y_max': ground_truth_box[3] if ground_truth_box else np.nan,
        'iou': iou,
        'bbox_quality': grade_iou(iou, status),
        'notes': safe_text(edge_decision.get('reason', '')),
    })

    final_canvas = image.copy()
    draw = ImageDraw.Draw(final_canvas)
    draw.rectangle([0, 0, final_canvas.width, 48], fill='black')
    draw.text(
        (10, 14),
        f'GPT guided OWL ViT | adjusted={adjustment_applied} | IoU={iou:.3f}',
        fill='white',
    )
    if ground_truth_box is not None:
        draw.rectangle([int(round(v)) for v in ground_truth_box], outline='green', width=6)
    if adjustment_applied:
        draw.rectangle([int(round(v)) for v in pre_adjustment_box], outline='orange', width=5)
    draw.rectangle([int(round(v)) for v in final_box], outline='cyan', width=8)
    final_canvas.save(FINAL_PREDICTION_DIR / f'{image_id}.png')

final_df = pd.DataFrame(final_rows)
initial_candidates_df = pd.DataFrame(initial_candidate_rows)
refined_candidates_df = pd.DataFrame(refined_candidate_rows)
selection_decisions_df = pd.DataFrame(selection_decision_rows)
edge_adjustments_df = pd.DataFrame(edge_adjustment_rows)

final_df.to_csv(FINAL_RESULTS_PATH, index=False)
initial_candidates_df.to_csv(METHOD_OUTPUT_DIR / 'owlvit_initial_candidates.csv', index=False)
refined_candidates_df.to_csv(METHOD_OUTPUT_DIR / 'owlvit_refined_candidates.csv', index=False)
selection_decisions_df.to_csv(METHOD_OUTPUT_DIR / 'gpt_candidate_decisions.csv', index=False)
edge_adjustments_df.to_csv(METHOD_OUTPUT_DIR / 'gpt_edge_adjustments.csv', index=False)

print('Method run complete.')
print('Final rows:', len(final_df))
print('Successful predictions:', int((final_df['status'] == 'success').sum()) if not final_df.empty else 0)
print('Final results:', FINAL_RESULTS_PATH)
display(final_df.head())

In [ ]:
# Final metrics for this method only. No baseline or oracle comparisons are loaded.

if final_df.empty:
    raise RuntimeError('No final result rows were generated.')

iou_values = pd.to_numeric(final_df['iou'], errors='coerce').fillna(0.0)
success_mask = final_df['status'].astype(str).eq('success')
adjustment_mask = final_df.get('adjustment_applied', pd.Series(False, index=final_df.index)).fillna(False).astype(bool)

method_metrics = pd.DataFrame([{
    'method_name': 'GPT guided OWL ViT with bounded refinement',
    'rows': len(final_df),
    'successful_predictions': int(success_mask.sum()),
    'failed_predictions': int((~success_mask).sum()),
    'mean_iou': float(iou_values.mean()),
    'median_iou': float(iou_values.median()),
    'success_rate_iou_025': float((iou_values >= SUCCESS_IOU_THRESHOLD).mean()),
    'strict_success_rate_iou_050': float((iou_values >= STRICT_IOU_THRESHOLD).mean()),
    'weak_or_better_rate_iou_010': float((iou_values >= WEAK_IOU_THRESHOLD).mean()),
    'gpt_adjustments_applied': int(adjustment_mask.sum()),
    'gpt_adjustment_rate': float(adjustment_mask.mean()),
    'mean_gpt_selection_confidence': float(pd.to_numeric(final_df.get('gpt_selection_confidence'), errors='coerce').mean()),
    'mean_gpt_edge_confidence': float(pd.to_numeric(final_df.get('gpt_edge_confidence'), errors='coerce').mean()),
}])

quality_counts = (
    final_df['bbox_quality']
    .value_counts(dropna=False)
    .rename_axis('bbox_quality')
    .reset_index(name='count')
)

method_metrics.to_csv(METRICS_DIR / 'method_metrics.csv', index=False)
quality_counts.to_csv(METRICS_DIR / 'bbox_quality_counts.csv', index=False)

print('Saved method metrics:', METRICS_DIR / 'method_metrics.csv')
print('Saved quality counts:', METRICS_DIR / 'bbox_quality_counts.csv')
display(method_metrics)
display(quality_counts)

# Notebook 04 output contract for Notebook 05.
expected_ids = set(benchmark['image_id'].astype(str))
result_ids = set(final_df['image_id'].astype(str))

if final_df['image_id'].astype(str).duplicated().any():
    raise ValueError('GPT guided OWL ViT results contain duplicate image IDs.')
if result_ids != expected_ids:
    raise ValueError(
        'Notebook 04 result IDs do not match the parsed benchmark. '
        f'Missing={sorted(expected_ids - result_ids)[:10]}, '
        f'extra={sorted(result_ids - expected_ids)[:10]}'
    )
if not FINAL_RESULTS_PATH.exists():
    raise FileNotFoundError(f'Notebook 04 did not create: {FINAL_RESULTS_PATH}')

contract_df = pd.DataFrame([{
    'result_path': str(FINAL_RESULTS_PATH.relative_to(PROJECT_ROOT)).replace('\\', '/'),
    'benchmark_path': str(PARSED_BENCHMARK_PATH.relative_to(PROJECT_ROOT)).replace('\\', '/'),
    'ground_truth_path': str(GT_PATH.relative_to(PROJECT_ROOT)).replace('\\', '/'),
    'expected_rows': len(expected_ids),
    'result_rows': len(final_df),
    'duplicate_image_ids': int(final_df['image_id'].astype(str).duplicated().sum()),
    'id_set_matches_benchmark': result_ids == expected_ids,
}])
contract_path = METHOD_RESULTS_DIR / 'notebook04_output_contract.csv'
contract_df.to_csv(contract_path, index=False)

print('Notebook 04 integration contract passed.')
print('Notebook 05 reads:', FINAL_RESULTS_PATH)
display(contract_df)


## Outputs

The notebook writes only this combined method:

```text
examples/outputs/gpt_guided_owlvit/
    gpt_guided_owlvit_results.csv
    owlvit_initial_candidates.csv
    owlvit_refined_candidates.csv
    gpt_candidate_decisions.csv
    gpt_edge_adjustments.csv

results/gpt_guided_owlvit/
    initial_candidates/
    refinement_regions/
    refined_candidates/
    pre_adjustment/
    edge_review/
    final_predictions/
    metrics/method_metrics.csv
    metrics/bbox_quality_counts.csv
```

Final screenshot colors:

- green: ground truth
- orange: selected box before GPT edge adjustment, shown only when an adjustment was applied
- cyan: final prediction